# IME júnior — 7 anos de dados financeiros: o que os números mostram

Análise de 2018 a 2025 a partir dos lançamentos financeiros da empresa júnior, unificados de 7 planilhas heterogêneas num pipeline Python (pandas). Nomes de clientes foram substituídos por códigos (`Cliente 001`, `Cliente 002`, ...) para preservar confidencialidade — os valores e padrões são reais.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/dataset_publico.csv", parse_dates=["data"])
df.head()

## Evolução de entrada e saída (2018-2025)

In [ ]:
mensal = df.set_index("data").groupby("tipo")["valor"].resample("ME").sum().unstack("tipo").fillna(0)
mensal["resultado"] = mensal["ENTRADA"] - mensal["SAÍDA"]

fig, ax = plt.subplots(figsize=(14, 5))
mensal[["ENTRADA", "SAÍDA"]].plot(ax=ax)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("Entrada x Saída mensal — IME júnior (2018-2025)")
ax.set_ylabel("R$")
plt.show()

### Por que 2025 destoa dos outros anos

O gráfico mostra uma queda acentuada de atividade financeira em 2025 em comparação aos anos anteriores — os picos mensais caem de R$ 14-16 mil (2018-2024) para no máximo ~R$ 4 mil, com novembro e dezembro praticamente zerados.

Essa queda não é um erro de dado: a gestão 2025 recebeu a empresa com o caixa devedor, e o foco do ano foi resolver trâmites internos e reorganizar as finanças, não captar novos projetos no ritmo histórico. O baixo volume reflete uma gestão de recuperação, não um declínio operacional comum.

## Sazonalidade: um padrão que não se confirmou

In [ ]:
df["mes_civil"] = df["data"].dt.month
sazonalidade = df.groupby(["mes_civil", "tipo"])["valor"].mean().unstack("tipo")
sazonalidade

Janeiro aparece com a maior entrada média do ano (R$ 1.125, mais que o dobro de qualquer outro mês) — à primeira vista, um padrão sazonal forte. Mas quebrando por ano, a história muda:

In [ ]:
janeiro = df[df["mes_civil"] == 1].copy()
janeiro["ano"] = janeiro["data"].dt.year
janeiro.groupby(["ano", "tipo"])["valor"].agg(["count", "mean", "std"])

### Conclusão: outlier, não sazonalidade

Em quase todo ano, o desvio padrão é maior que a própria média — sinal de que poucos lançamentos grandes distorcem a média de grupos pequenos (3 a 9 lançamentos por mês). O caso mais extremo é 2022, onde janeiro teve um único lançamento (R$ 4.000) que sozinho definiu "a média do mês". Em 2025, não há nenhuma entrada registrada em janeiro.

O pico de janeiro não representa um padrão sazonal confiável do calendário universitário — representa poucos anos com poucos lançamentos, alguns coincidentemente grandes. Um bom lembrete de checar antes de acreditar num padrão bonito.

## Ticket médio e volume de projetos por gestão

In [ ]:
projetos = df[(df["tipo"] == "ENTRADA") & (df["categoria"] == "projeto")]
por_projeto = projetos.groupby(["gestao", "cliente_projeto"])["valor"].sum().reset_index()

ticket_medio = por_projeto.groupby("gestao")["valor"].agg(["count", "mean", "median"])
ticket_medio

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ticket_medio["median"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Ticket médio (mediana) por gestão — projetos fechados")
ax.set_ylabel("R$")
ax.set_xlabel("Gestão")
for i, (gestao, row) in enumerate(ticket_medio.iterrows()):
    ax.text(i, row["median"] + 50, f"n={int(row['count'])}", ha="center", fontsize=9)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Queda no volume de projetos fechados

O número de projetos distintos fechados por gestão caiu de forma acentuada: 28 projetos na gestão 18-19, para apenas 3-4 nos últimos anos. Isso é uma tendência mais preocupante que o valor do ticket médio em si — mesmo nos anos em que o ticket médio ficou estável, a empresa fechou muito menos negócios. Diferente de um gasto pontual, isso aponta pra um problema de captação, não de precificação.

## Concentração de clientes

In [ ]:
receita_por_cliente = por_projeto.groupby("cliente_projeto")["valor"].sum().sort_values(ascending=False)
receita_total = receita_por_cliente.sum()
top10 = receita_por_cliente.head(10)

print(f"Receita total (7 anos): R$ {receita_total:,.2f}")
print(f"Número de clientes distintos: {receita_por_cliente.shape[0]}")
print(f"Top 10 clientes somam: R$ {top10.sum():,.2f} ({top10.sum()/receita_total*100:.1f}% do total)")
top10

Em 7 anos, a empresa faturou com quase 100 clientes/projetos distintos, mas os **10 maiores concentram 45,6% de toda a receita**. Um único cliente sozinho representa **15,1%** de tudo que a empresa já faturou.

Isso confirma, com número, o que a diferença entre média e mediana do ticket já sugeria: a receita depende fortemente de poucos clientes grandes — um risco de concentração real para o caixa da empresa.

## Composição de despesas

In [ ]:
despesas = df[(df["tipo"] == "SAÍDA") & (df["categoria"] != "aplicação automática")]
composicao = despesas.groupby("categoria")["valor"].sum().sort_values(ascending=False)

top10_desp = composicao.head(10)
outros = composicao.iloc[10:].sum()
grafico_dados = pd.concat([top10_desp, pd.Series({"outros": outros})]).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
grafico_dados.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Composição de despesas — top 10 categorias + outros (2018-2025)")
ax.set_xlabel("R$")
plt.tight_layout()
plt.show()

### Imersão: a maior despesa histórica e o déficit herdado por 2025

**Imersão** é, disparado, a maior categoria de gasto da empresa em 7 anos (R$ 101.119,98 — quase o dobro da segunda colocada). Cruzando com o resultado por gestão: as três gestões anteriores a 2025 tiveram resultado negativo em sequência, e são exatamente os três anos de maior gasto com imersão (66% de todo o gasto histórico nessa categoria).

A imersão era a antiga festa de confraternização da empresa — um gasto recorrente e alto que, mantido por três gestões seguidas de resultado negativo, consumiu o caixa. A gestão 2025 não gerou esse déficit: herdou o acumulado das anteriores, entrando com caixa devedor mesmo tendo, ela própria, fechado o ano com resultado positivo e **sem nenhum gasto com imersão registrado** — um corte direto adotado para estancar o problema.